# Collaudo del catalogo · lookahead e degenerazione

Questo notebook controlla **tutte le condizioni registrate** (entry, exit, filtri) e gli indicatori.

- **Sezione A** — collaudo del collaudo: condizioni-trappola costruite apposta. Se A non passa, B e C non valgono.
- **Sezione B** — lookahead su tutto il catalogo, su dati sintetici. Esito: promosso / bocciato.
- **Sezione C** — degenerazione sui dati veri (EURUSD dal repo). Esito: una tabella da leggere.

**Come si esegue:** in Colab, menu *Runtime → Esegui tutto*. Tempo totale: 3–5 minuti, quasi tutto nella sezione B.

Il notebook clona il repo da GitHub: i file `engine/indicatori.py` e `engine/collaudo_catalogo.py` devono essere già caricati sul repo.

## 0 · Preparazione

Scarica il repo (o lo aggiorna, se c'è già) e installa TA-Lib. Alla fine stampa le versioni: se qualcosa va storto, sono le prime cose da guardare.

In [ ]:
import os
REPO = "Trading-System-Creator_V4"
if not os.path.exists(REPO):
    !git clone -q https://github.com/matti410/Trading-System-Creator_V4.git
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q ta-lib

import sys, numpy as np, pandas as pd, talib
print(f"python {sys.version.split()[0]} · pandas {pd.__version__} · numpy {np.__version__} · ta-lib {talib.__version__}")

## 1 · Il catalogo

`registra_catalogo()` chiama tutte le funzioni `registra_*`. **Una condizione nuova in un file esistente è coperta da sola.** Un *file* di condizioni nuovo va aggiunto con una riga alla lista `REGISTRAZIONI`.

In [ ]:
import contextlib, io
from engine import registry as R
from engine.indicatori import aggiungi_indicatori
from engine.collaudo_catalogo import (condizioni_registrate, verifica_lookahead,
                                      verifica_degenerazione, mercato_sintetico)
from engine.broker_tz_diagnostic import to_utc_index
from helpers import _evento
import entry_long, entry_short, entry_metro, exit_long, exit_short
import filter_conditions, vwap_regime_filter_conditions

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

REGISTRAZIONI = [
    entry_long.registra_trigger_long,
    entry_short.registra_trigger_short,
    lambda: entry_metro.registra_trigger_metro(anche_short=True),
    exit_long.registra_exit_long,
    exit_short.registra_exit_short,
    filter_conditions.registra_filtri,
    vwap_regime_filter_conditions.registra_filtri_vwap,
]

def registra_catalogo():
    R.clear_registry()
    with contextlib.redirect_stdout(io.StringIO()):   # niente righe di log
        for f in REGISTRAZIONI:
            f()

registra_catalogo()
catalogo = condizioni_registrate()
print(f"{len(catalogo)} condizioni registrate")
catalogo["tipo"].value_counts()

## A1 · Collaudo del test di lookahead

Si registrano condizioni-trappola **con lookahead messo apposta**, due condizioni pulite e una mai vera. Il test deve riconoscerle tutte. C'è anche un indicatore-trappola (`media_centrata`), per verificare che il test ricalcoli gli indicatori a ogni taglio.

In fondo: `SEZIONE A1: n/n`. Se non sono tutte giuste, il notebook si ferma.

In [ ]:
R.clear_registry()

# --- trappole: ognuna guarda avanti in un modo diverso -------------------
def t_shift_meno_uno(df):      # legge la chiusura della barra dopo
    return _evento(df["Close"].shift(-1) > df["Close"])

def t_shift_raro(df):          # come sopra, ma rara: serve ai tagli mirati
    salto = df["Close"].shift(-1) - df["Close"]
    return _evento(salto > 3 * salto.abs().rolling(500).mean())

def t_massimo_giorno(df):      # il massimo del giorno si conosce solo a fine giorno
    return df["High"] == df["High"].groupby(df.index.date).transform("max")

def t_rolling_centrata(df):    # media centrata: usa 4 barre future
    return df["Close"] > df["Close"].rolling(9, center=True).mean()

def t_quantile_globale(df):    # soglia calcolata su tutto lo storico
    return df["atr"] > df["atr"].quantile(0.7)

def t_prossimo_timestamp(df):  # "la barra dopo non c'e'": ultima prima del weekend
    tempi = pd.Series(df.index, index=df.index)
    return tempi.diff().shift(-1) > pd.Timedelta(hours=1)

_CACHE_TRAPPOLA = {}
def t_cache(df):               # lookahead nascosto dietro una cache sull'indice
    k = (len(df), df.index[0], df.index[-1])
    if k not in _CACHE_TRAPPOLA:
        _CACHE_TRAPPOLA[k] = df["High"].groupby(df.index.date).transform("max")
    return df["High"] >= _CACHE_TRAPPOLA[k]

def t_usa_indicatore(df):      # pulita in se', ma legge un indicatore che guarda avanti
    return df["Close"] > df["media_centrata"]

# --- pulite e mai vera ----------------------------------------------------
def t_pulita_evento(df):
    c, e = df["Close"], df["ema20"]
    return _evento((c > e) & (c.shift(1) <= e.shift(1)))

def t_pulita_filtro(df):
    return df["Close"] > df["Close"].rolling(50).mean()

def t_mai_vera(df):
    return df["Close"] < 0

def prepara_con_trappola(df):
    d = aggiungi_indicatori(df)
    d["media_centrata"] = d["Close"].rolling(9, center=True).mean()
    return d

R.register_entry("T_SHIFT_MENO_UNO", 1)(t_shift_meno_uno)
R.register_entry("T_SHIFT_RARO", 1)(t_shift_raro)
R.register_entry("T_PULITA_EVENTO", 1)(t_pulita_evento)
for nome, f in [("T_MASSIMO_GIORNO", t_massimo_giorno), ("T_ROLLING_CENTRATA", t_rolling_centrata),
                ("T_QUANTILE_GLOBALE", t_quantile_globale), ("T_PROSSIMO_TIMESTAMP", t_prossimo_timestamp),
                ("T_CACHE", t_cache), ("T_USA_INDICATORE", t_usa_indicatore),
                ("T_PULITA_FILTRO", t_pulita_filtro), ("T_MAI_VERA", t_mai_vera)]:
    R.register_filter(nome)(f)

ATTESI_A1 = {
    "T_SHIFT_MENO_UNO": "LOOKAHEAD", "T_SHIFT_RARO": "LOOKAHEAD",
    "T_MASSIMO_GIORNO": "LOOKAHEAD", "T_ROLLING_CENTRATA": "LOOKAHEAD",
    "T_QUANTILE_GLOBALE": "LOOKAHEAD", "T_PROSSIMO_TIMESTAMP": "LOOKAHEAD",
    "T_CACHE": "LOOKAHEAD", "T_USA_INDICATORE": "LOOKAHEAD",
    "media_centrata": "LOOKAHEAD",
    "T_PULITA_EVENTO": "OK", "T_PULITA_FILTRO": "OK", "T_MAI_VERA": "NON_ESERCITATA",
    # gli indicatori veri devono uscire puliti
    **{c: "OK" for c in ("rsi", "macd", "macd_signal", "macd_hist", "ema20", "ema50",
                         "zlema50", "atr", "realized_vol", "adx", "vwap")},
}

rep_a1 = verifica_lookahead(mercato_sintetico(), prepara=prepara_con_trappola, verbose=False)
ottenuti = rep_a1.set_index("nome")["esito"]
verifica_a1 = pd.DataFrame({"atteso": pd.Series(ATTESI_A1),
                            "ottenuto": ottenuti.reindex(list(ATTESI_A1))})
verifica_a1["giusto"] = verifica_a1["atteso"] == verifica_a1["ottenuto"]
display(verifica_a1)
giusti = int(verifica_a1["giusto"].sum())
print(f"SEZIONE A1: {giusti}/{len(verifica_a1)}")
assert giusti == len(verifica_a1), "Il test di lookahead NON riconosce le trappole: le sezioni B e C non valgono."

## A2 · Collaudo del test di degenerazione

Condizioni costruite a mano, con risultato noto: sempre vera, sempre falsa, quasi sempre vera, rara, uno *stato* al posto di un evento, una normale, una a grappoli, una che non restituisce booleani, una che va in errore, e l'exit «nessuna uscita» che deve risultare `ATTESA`.

Per l'entry a grappoli il risultato è noto esattamente: gruppi di 4 eventi a 2 barre di distanza, quindi `quota_in_grappolo` = 0,75. Con `distanza_grappolo=1` gli stessi eventi risultano tutti separati (quota 0).

In [ ]:
R.clear_registry()
df_a2 = aggiungi_indicatori(mercato_sintetico())
N = len(df_a2)

def maschera(df, posizioni):
    m = np.zeros(len(df), dtype=bool)
    m[np.asarray(list(posizioni), dtype=int)] = True
    return pd.Series(m, index=df.index)

GRUPPI = range(100, N - 10, 200)          # un grappolo ogni 200 barre
R.register_filter("D_SEMPRE_VERA")(lambda df: pd.Series(True, index=df.index))
R.register_filter("D_SEMPRE_FALSA")(lambda df: pd.Series(False, index=df.index))
R.register_filter("D_QUASI_SEMPRE")(lambda df: ~maschera(df, range(0, len(df), 50)))   # 98%
R.register_filter("D_NON_BOOLEANA")(lambda df: df["Close"])
R.register_filter("D_ERRORE")(lambda df: df["colonna_che_non_esiste"] > 0)
R.register_entry("D_RARA", 1)(lambda df: maschera(df, range(500, 500 + 10 * 300, 300)))   # 10 eventi
R.register_entry("D_STATO", 1)(lambda df: maschera(df, [p + k for p in range(0, len(df) - 3, 100) for k in range(3)]))
R.register_entry("D_OK", 1)(lambda df: maschera(df, range(0, len(df), 50)))
R.register_entry("D_GRAPPOLO", 1)(lambda df: maschera(df, [g + k for g in GRUPPI for k in (0, 2, 4, 6)]))
R.register_exit("X0_NO_EXIT", 1)(lambda df: pd.Series(False, index=df.index))

rep_a2 = verifica_degenerazione(df_a2, verbose=False).set_index("nome")
rep_a2_d1 = verifica_degenerazione(df_a2, distanza_grappolo=1, verbose=False).set_index("nome")

ATTESI_A2 = {
    "D_SEMPRE_VERA": "SEMPRE_VERA", "D_SEMPRE_FALSA": "SEMPRE_FALSA",
    "D_QUASI_SEMPRE": "QUASI_SEMPRE_VERA", "D_NON_BOOLEANA": "NON_BOOLEANA",
    "D_ERRORE": "ERRORE", "D_RARA": "RARA", "D_STATO": "STATO",
    "D_OK": "OK", "D_GRAPPOLO": "OK", "X0_NO_EXIT": "ATTESA",
}
verifica_a2 = pd.DataFrame({"atteso": pd.Series(ATTESI_A2),
                            "ottenuto": rep_a2["esito"].reindex(list(ATTESI_A2))})
controlli = {
    "D_GRAPPOLO: grappoli = numero di gruppi": rep_a2.loc["D_GRAPPOLO", "grappoli"] == len(GRUPPI),
    "D_GRAPPOLO: quota_in_grappolo = 0,75": abs(rep_a2.loc["D_GRAPPOLO", "quota_in_grappolo"] - 0.75) < 1e-12,
    "D_GRAPPOLO con distanza 1: quota 0": rep_a2_d1.loc["D_GRAPPOLO", "quota_in_grappolo"] == 0,
    "D_OK: nessun grappolo": rep_a2.loc["D_OK", "quota_in_grappolo"] == 0,
    "D_RARA: 10 eventi": rep_a2.loc["D_RARA", "eventi"] == 10,
    "D_STATO: barre consecutive contate": rep_a2.loc["D_STATO", "barre_consecutive"] == 2 * rep_a2.loc["D_STATO", "eventi"],
}
for nome, ok in controlli.items():
    verifica_a2.loc[nome] = ["vero", "vero" if ok else "falso"]
verifica_a2["giusto"] = verifica_a2["atteso"] == verifica_a2["ottenuto"]
display(verifica_a2)
giusti = int(verifica_a2["giusto"].sum())
print(f"SEZIONE A2: {giusti}/{len(verifica_a2)}")
assert giusti == len(verifica_a2), "Il test di degenerazione NON classifica bene i casi noti."

## B · Lookahead su tutto il catalogo

Dati sintetici (quattro mesi di random walk M15 con i cambi d'ora d'autunno). Per ogni condizione: 104 tagli comuni (uno per ogni quarto d'ora, più i bordi del weekend) e fino a 5 tagli mirati sulle barre in cui la condizione è vera. Tempo: circa 2 minuti.

**Come si legge**
- `LOOKAHEAD` → la condizione cambia valore quando si toglie il futuro. La colonna `primo_caso` dice dove.
- `ERRORE` → la condizione non gira; il messaggio è nella colonna `errore`.
- `NON_ESERCITATA` → mai vera su questi dati, quindi non verificabile. Le due exit `X0_…_NO_EXIT` sono sempre false per costruzione: per loro è normale.
- Le righe `indicatore` sono gli indicatori di `engine/indicatori.py`, verificati da soli.

Il test fallisce solo per `LOOKAHEAD` o `ERRORE`.

In [ ]:
registra_catalogo()
rep_b = verifica_lookahead(mercato_sintetico())
problemi_b = rep_b[rep_b["esito"].isin(["LOOKAHEAD", "ERRORE"])]
print(f"\nSEZIONE B: {'SUPERATA' if problemi_b.empty else 'NON SUPERATA'} — "
      f"{len(problemi_b)} condizioni con LOOKAHEAD o ERRORE")
display(rep_b[rep_b["esito"] != "OK"])

La tabella completa, tutte le righe:

In [ ]:
rep_b

## C · Degenerazione sui dati veri

EURUSD M15 dal repo, convertito in UTC con la stessa regola del notebook principale, indicatori da `engine/indicatori.py`. Nessun promosso/bocciato: la degenerazione dipende dal dataset.

**Come si legge**
- `SEMPRE_FALSA` → mai vera: di solito è un bug (colonna sbagliata, soglia impossibile).
- `SEMPRE_VERA` / `QUASI_SEMPRE_VERA` → vera sul 95% delle barre o più: non filtra.
- `RARA` → meno di 30 occorrenze (per le entry si contano gli eventi, per filtri ed exit le barre vere).
- `STATO` → un'entry vera su barre consecutive: manca `_evento`.
- `ATTESA` → le exit «nessuna uscita», sempre false per costruzione.
- `grappoli` / `quota_in_grappolo` (solo entry) → quanti eventi cadono entro `distanza_grappolo` barre dal precedente. Informativo, non cambia l'esito.

Le soglie si cambiano nella chiamata: `verifica_degenerazione(df_vero, min_occorrenze=30, copertura_max=0.95, distanza_grappolo=8)`.

In [ ]:
df_vero = pd.read_csv("EURUSD_M15.csv", index_col="Date")
df_vero = to_utc_index(df_vero, "A_US_DST (NY+7h)", on_dst_gap="drop", verbose=False)
df_vero = aggiungi_indicatori(df_vero)

rep_c = verifica_degenerazione(df_vero, min_occorrenze=30, copertura_max=0.95, distanza_grappolo=8)
display(rep_c[rep_c["esito"] != "OK"])

La tabella completa, ordinata per tipo e copertura:

In [ ]:
rep_c.sort_values(["tipo", "copertura"]).reset_index(drop=True)